# Two factors, one control decision

Runnable companion to `explainer_causal_factors_two_factor.html` (the write-up carries the
algebra; this notebook carries the numbers). The estimand is the causal loading of the return
`X` on factor `F2`. A second factor `F1` is a candidate control whose causal role decides
whether it belongs in the regression.

Each section draws one Monte Carlo sample at seed 42 and reads off two loadings: omit `F1`, then
control for `F1`. The gap between them, and which way it points, is the whole lesson. No latent
driver acts on `X` directly: confounding here always comes from a variable that reaches the return.

Run top to bottom. Light compute (numpy plus one 5 fold cross fit), well under a minute.

In [ ]:
import numpy as np

SEED = 42
N = 1_000_000  # Monte Carlo sample size; large enough that the loadings read off cleanly


def set_seed(seed: int = SEED) -> np.random.Generator:
    """Seeded NumPy Generator. Seed is explicit and logged, per the reproducibility rule."""
    return np.random.default_rng(seed)


def ols_slope(y, *regressors):
    """OLS slope on the FIRST regressor; remaining regressors are controls. Intercept added."""
    R = np.column_stack(list(regressors) + [np.ones(len(y))])
    coef, *_ = np.linalg.lstsq(R, y, rcond=None)
    return coef[0]

## 1. F1 as a confounder (F1 -> F2, F1 -> X)

Omitting `F1` leaves the backdoor `F2 <- F1 -> X` open, so the loading soaks up part of `F1`'s
effect and lands away from the truth (upward here, since both arrows are positive). Controlling
for `F1` closes the backdoor and recovers the true loading.

**Why these numbers (closed form).** With `Var(F1) = 1`, the DGP gives `Var(F2) = a^2 + 1` and
`Cov(F2, F1) = a`.

- *Omit F1.* The slope is `Cov(X, F2) / Var(F2)`. Since `Cov(X, F2) = b*Var(F2) + c*Cov(F1, F2)
  = b*(a^2+1) + c*a`, the slope equals `b + a*c/(a^2+1) = 1 + (0.8)(0.7)/1.64 = 1.3415`. The extra
  term `a*c/(a^2+1)` is the backdoor leaking in.
- *Control F1.* Here `X = b*F2 + c*F1 + noise` with the noise independent of both regressors, so
  population OLS returns the true pair `(b, c)` exactly: the slope on `F2` is `b = 1.0`.

In [ ]:
rng = set_seed()
# Graph: F1 -> F2 (a) and F1 -> X (c) make F1 a confounder; F2 -> X (b) is the true loading.
b, a, c = 1.0, 0.8, 0.7
F1 = rng.standard_normal(N)
F2 = a * F1 + rng.standard_normal(N)
X = b * F2 + c * F1 + rng.standard_normal(N)

biased = ols_slope(X, F2)         # backdoor F2 <- F1 -> X left open
recovered = ols_slope(X, F2, F1)  # F1 controlled, backdoor closed

# Population closed forms the Monte Carlo converges to (derivation in the cell above).
omit_cf = b + a * c / (a**2 + 1)  # true loading + omitted-variable bias a*c/Var(F2)
ctrl_cf = b                       # X is linear in (F2, F1) with independent noise -> exact
print(f"omit F1: {biased:.4f}  (closed form {omit_cf:.4f})  -> biased (backdoor open)")
print(f"ctrl F1: {recovered:.4f}  (closed form {ctrl_cf:.4f})  -> recovers true loading b = {b:.1f}")

## 2. F1 as a collider (F2 -> F1, X -> F1)

No backdoor exists, so `F2` alone is already unbiased. Conditioning on the collider `F1`
opens the spurious path `F2 -> F1 <- X` and biases the slope toward (and past) zero.

**Why these numbers (closed form).** Write `F1` in reduced form:
`F1 = a*F2 + c*X + e1 = (a + c*b)*F2 + c*eX + e1`, so `p := Cov(F2, F1) = a + c*b`.

- *Omit F1.* No backdoor, so the slope is `Cov(X, F2) / Var(F2) = b = 1.0`.
- *Control F1.* The regressors `(F2, F1)` have covariance matrix `[[1, p], [p, p^2 + c^2 + 1]]`
  with determinant `det = c^2 + 1`. With `Cov(F2, X) = b` and `Cov(F1, X) = p*b + c
  = a*b + c*(b^2+1)`, Cramer's rule on the `F2` coordinate gives
  `slope = [(p^2 + c^2 + 1)*b - p*(a*b + c*(b^2+1))] / (c^2+1)`. With `a=0.8, c=0.7, b=1`:
  `p = 1.5`, `det = 1.49`, slope `= 0.44/1.49 = 0.2953`. Conditioning on the collider drags the
  loading from `1.0` down to about `0.30`.

In [ ]:
rng = set_seed()
# Graph: F2 -> F1 (a) and X -> F1 (c) make F1 a collider; F2 -> X (b) is the true loading.
b, a, c = 1.0, 0.8, 0.7
F2 = rng.standard_normal(N)
X = b * F2 + rng.standard_normal(N)
F1 = a * F2 + c * X + rng.standard_normal(N)

unbiased = ols_slope(X, F2)       # no backdoor: F2 alone is already correct
biased = ols_slope(X, F2, F1)     # conditioning on the collider opens F2 -> F1 <- X

# Population closed forms (derivation in the cell above). F1 = p*F2 + c*eX + e1, p = Cov(F2, F1).
p = a + c * b
det = c**2 + 1  # determinant of the 2x2 regressor covariance matrix
omit_cf = b     # no backdoor to open
ctrl_cf = ((p**2 + c**2 + 1) * b - p * (a * b + c * (b**2 + 1))) / det  # Cramer's rule on F2
print(f"omit F1: {unbiased:.4f}  (closed form {omit_cf:.4f})  -> unbiased (no backdoor to close)")
print(f"ctrl F1: {biased:.4f}  (closed form {ctrl_cf:.4f})  -> biased toward zero (collider opened)")

## 3. F1 as a mediator (F2 -> F1 -> X)

Now `F1` lies on the causal path. Omitting it gives the **total** effect of `F2`; controlling
for it gives the **direct** effect only. Neither is wrong, but they answer different questions.

In [ ]:
rng = set_seed()
m, b_dir, p = 0.7, 0.6, 0.5
F2 = rng.standard_normal(N)
F1 = m * F2 + rng.standard_normal(N)
X = b_dir * F2 + p * F1 + rng.standard_normal(N)

print(f"total  effect (omit F1): {ols_slope(X, F2):.4f}")
print(f"direct effect (ctrl F1): {ols_slope(X, F2, F1):.4f}")

## 4. F1 as a neutral predictor (F1 -> X, F1 independent of F2)

`F1` is neither a confounder nor a collider, so the loading is unbiased either way. Including
`F1` is still worth it: it absorbs residual variance in `X` and shrinks the standard error on
the loading.

In [ ]:
rng = set_seed()
beta, q = 1.0, 0.8
F2 = rng.standard_normal(N)
F1 = rng.standard_normal(N)
X = beta * F2 + q * F1 + rng.standard_normal(N)


def ols_slope_se(y, *regressors):
    R = np.column_stack(list(regressors) + [np.ones(len(y))])
    coef, *_ = np.linalg.lstsq(R, y, rcond=None)
    resid = y - R @ coef
    s2 = resid @ resid / (len(y) - R.shape[1])
    cov = s2 * np.linalg.inv(R.T @ R)
    return coef[0], np.sqrt(cov[0, 0])


b_no, se_no = ols_slope_se(X, F2)
b_ct, se_ct = ols_slope_se(X, F2, F1)
print(f"omit F1: b={b_no:.4f}  se={se_no:.5f}")
print(f"ctrl F1: b={b_ct:.4f}  se={se_ct:.5f}   -> same loading, SE x{se_ct / se_no:.2f}")

## 5. A hidden driver that never touches the return (U -> F1, U -> F2, U -/-> X)

A latent `U` drives both factors but not the return. It makes the factors correlated, yet the
loadings stay unbiased: this is collinearity (a precision cost), not confounding.

In [ ]:
rng = set_seed()
g1, g2, b1, b2 = 0.8, 0.4, 1.0, 0.5
U = rng.standard_normal(N)
F1 = g1 * U + rng.standard_normal(N)
F2 = g2 * U + rng.standard_normal(N)
X = b1 * F1 + b2 * F2 + rng.standard_normal(N)  # U does NOT enter X

R = np.column_stack([F1, F2, np.ones(N)])
coef, *_ = np.linalg.lstsq(R, X, rcond=None)
corr = np.corrcoef(F1, F2)[0, 1]
print(f"OLS loadings (F1, F2): ({coef[0]:.4f}, {coef[1]:.4f})   vs true (1.000, 0.500)")
print(f"corr(F1, F2) = {corr:.4f}   -> correlated regressors, unbiased loadings")

## 6. The DML dividing line: estimation does not manufacture identification

Cross fitted DML (K=5, gradient boosted nuisances) for the loading on `F2` under the confounder
data generating process. The method recovers the loading only when the controls `W` actually
contain the confounder; with a noisy proxy or irrelevant controls it is precise but wrong.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold

rng = set_seed()
n = 40_000
F1 = rng.standard_normal(n)
F2 = 0.8 * F1 + rng.standard_normal(n)
X = 1.0 * F2 + 0.7 * F1 + rng.standard_normal(n)  # true loading on F2 is 1.0
F1_proxy = F1 + rng.standard_normal(n)
W_irrelevant = rng.standard_normal(n)


def dml_loading(*control_cols):
    W = np.column_stack(control_cols)
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    rF2 = np.zeros(n)
    rX = np.zeros(n)
    for tr, te in kf.split(W):
        mF2 = HistGradientBoostingRegressor(random_state=SEED).fit(W[tr], F2[tr])
        rF2[te] = F2[te] - mF2.predict(W[te])
        mX = HistGradientBoostingRegressor(random_state=SEED).fit(W[tr], X[tr])
        rX[te] = X[te] - mX.predict(W[te])
    return float((rF2 @ rX) / (rF2 @ rF2))


print(f"W contains F1 (confounder): {dml_loading(F1):.4f}   -> recovers the loading")
print(f"W is a noisy proxy of F1:   {dml_loading(F1_proxy):.4f}   -> still biased")
print(f"W is irrelevant:            {dml_loading(W_irrelevant):.4f}   -> the biased OLS projection")

## Summary

| F1's role | omit F1 | control F1 | decision |
|---|---|---|---|
| confounder | biased (away from truth) | unbiased | control |
| collider | unbiased | biased (toward zero) | do not control |
| mediator | total effect | direct effect | depends on estimand |
| neutral predictor | unbiased | unbiased, smaller SE | optional (precision) |
| driver of factors only | unbiased | n/a (F1 is itself a factor) | no action; collinearity only |

The two middle columns describe the loading on `F2`; run the cells above for the numbers at
seed 42. Control a confounder, never a collider, mind the mediator. Across the first four rows
the regression fit looks the same; only the causal graph tells them apart.